# WildSpeech V4 — Speech–Text–Acoustic Multimodal Intelligence (Google Colab)

This notebook drives `wildspeech.v4.pipeline.V4Pipeline` from the WildSpeech repository. Nothing in it re-implements the pipeline: every cell calls the package.

**What it does:** mounts Drive, clones WildSpeech at a pinned commit, installs it in editable mode, imports the completed external ASR export (`all_models_predictions.parquet`), validates audio/text/metadata joins, reuses or extracts acoustic features, builds recognized-text / disagreement / (measured) phonetic / acoustic student representations, evaluates M0–M8 on identical student-disjoint folds with ablations, negative controls and robustness schemes, tests H1–H10, produces figures, selects the champion from validated out-of-fold comparisons, refits it on all labelled students and writes a validated submission — all under `CONFIG['output_root']` on Drive.

**Before the first run**
1. Fill `V4_OVERRIDES['asr_import']['permission_review']` (cell 1). Without a completed review every ASR-derived family is `NOT_EXECUTED` (the tabular and acoustic experiments still run). The ASR predictions are imported with provenance `external_colab_asr`; they are never relabelled as local P24 output.
2. Run with `run_mode='PREPARE'` first and read `asr_import/asr_schema_report.json`. If a role (model / filename / transcript / phones ...) was not detected, set `V4_OVERRIDES['asr_import']['column_map']`.
3. Optional: upload a copy of a compatible local `cache/features` folder to Drive and list it in `V4_OVERRIDES['acoustic']['search_dirs']`; only fingerprint-compatible families are reused.

**Modes** (`CONFIG['run_mode']`): `PREPARE` → `FEATURES` → `EXPERIMENTS` → `SUBMISSION`, or `FULL` (all), or `SMOKE` (subsample, 1 repeat, no submission). Every step resumes from the artefacts already on Drive.

No empirical claim exists until these cells have been executed; the report is built from run artefacts only.

In [ ]:
# 1. Configuration
CONFIG = {
    'repo': 'mybenkhadda/WildSpeech',
    'ref': 'wildspeech-v4-multimodal',     # branch holding the V4 code (use 'main' once merged)
    'pinned_commit': '17222a0cc700146a62b675fcb2649c0963133c45',  # V4 code commit (set None to follow 'ref')
    'project_subdir': 'WildSpeech_GPU',

    'data_root': '/content/drive/MyDrive/WildSpeech',
    'asr_root': '/content/drive/MyDrive/WildSpeech/WildSpeech_ASR_Exports',
    'asr_filename': 'all_models_predictions.parquet',
    'asr_drive_file_id': '12SzHSmkNbeoS_mO1L80uEd6A2k4jH-8V',   # used ONLY if the mounted file is missing

    'output_root': '/content/drive/MyDrive/WildSpeech/WildSpeech_V4',
    'local_work_root': '/content/wildspeech_v4',

    'run_mode': 'FULL',                    # FULL | PREPARE | FEATURES | EXPERIMENTS | SUBMISSION | SMOKE
    'max_clips': None,
    'splits': ['train', 'test'],

    'reuse_existing_features': True,
    'compute_missing_acoustic_features': True,
    'rerun_asr': False,                    # V4 imports the completed export; local ASR is not re-run
    'train_models': True,
    'generate_submission': True,

    'cv_folds': 5,
    'cv_repeats': 3,
    'seed': 2026,

    'use_gpu': True,
    'resume': True,
    'save_checkpoints': True,
}

# V4 settings (merged over configs/v4.yaml). Fill the permission review before importing ASR predictions.
V4_OVERRIDES = {
    'asr_import': {
        'permission_review': {
            'reviewed': False,           # True only after reviewing licences / competition rules of every exported model
            'reviewer': '',
            'date': '',                  # YYYY-MM-DD
            'statement': '',
            'models': [],                # [] = all exported models; otherwise only these model ids
        },
        'column_map': {},                # e.g. {'model_id': 'model', 'filename': 'audio_name', 'transcript': 'text'}
        'wide_model_columns': {},        # wide exports: {model_id: transcript_column}
    },
    'acoustic': {'search_dirs': []},     # e.g. ['/content/drive/MyDrive/WildSpeech/local_cache/features']
    'hier_net': {'enabled': False},      # M8 (evaluated only if an M1-M7 row is a GENUINE_GAIN)
}
print('run_mode =', CONFIG['run_mode'])

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
for k in ('data_root', 'asr_root'):
    print(f'{k:10s}', CONFIG[k], '->', 'found' if os.path.isdir(CONFIG[k]) else 'MISSING')
print('ASR export:', os.path.exists(os.path.join(CONFIG['asr_root'], CONFIG['asr_filename'])))

In [ ]:
# 3. Clone WildSpeech at a pinned revision
import pathlib, subprocess, sys

def sh(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

REPO_DIR = pathlib.Path('/content/WildSpeech')
if not (REPO_DIR / '.git').exists():
    sh(f"git clone https://github.com/{CONFIG['repo']}.git {REPO_DIR}")
sh(f'git -C {REPO_DIR} fetch --all --tags --prune')
target = CONFIG.get('pinned_commit') or f"origin/{CONFIG['ref']}"
sh(f'git -C {REPO_DIR} checkout --detach {target}')
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD']).decode().strip()
CONFIG['resolved_commit'] = COMMIT
print('WildSpeech commit:', COMMIT)
if not CONFIG.get('pinned_commit'):
    print(f"NOTE: floating ref {CONFIG['ref']!r}. For a reproducible run set CONFIG['pinned_commit'] = '{COMMIT}'")
PROJECT = REPO_DIR / CONFIG['project_subdir']
assert (PROJECT / 'src' / 'wildspeech' / 'v4' / 'pipeline.py').exists(), 'this revision has no V4 code'

In [ ]:
# 4. Install the package (editable) and the missing Colab-compatible dependencies.
#    No ASR foundation checkpoint is installed or loaded: the export is re-read, never re-inferred.
sh(f'{sys.executable} -m pip install -q -e {PROJECT}')
missing = []
for mod, pkg in [('catboost', 'catboost'), ('lightgbm', 'lightgbm'), ('xgboost', 'xgboost'), ('pygam', 'pygam'),
                 ('soxr', 'soxr'), ('pywt', 'PyWavelets'), ('soundfile', 'soundfile'), ('shap', 'shap'),
                 ('gdown', 'gdown'), ('g2p_en', 'g2p_en'), ('nltk', 'nltk'), ('yaml', 'pyyaml'), ('pyarrow', 'pyarrow')]:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    sh(f"{sys.executable} -m pip install -q {' '.join(missing)}")
import nltk
for res in ('cmudict', 'averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger'):
    try:
        nltk.download(res, quiet=True)
    except Exception as e:
        print('nltk', res, e)
if str(PROJECT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT / 'src'))          # editable install without a kernel restart
import wildspeech
print('wildspeech from', wildspeech.__file__)

In [ ]:
# 5. Build the V4 pipeline (resolves Drive paths, builds the run context, writes reports/run_config.json)
import pandas as pd
from IPython.display import Image, Markdown, display
from wildspeech.v4.config import resolve_asr_file
from wildspeech.v4.pipeline import V4Pipeline

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pipe = V4Pipeline(CONFIG, project_root=PROJECT, v4_overrides=V4_OVERRIDES)
print('data dir :', pipe.ctx.paths.data_dir)
display(pd.Series({k: str(v) for k, v in pipe.ctx.paths.files.items()}, name='discovered files'))
asr_path, asr_info = resolve_asr_file(pipe.run_cfg, allow_download=False)
print('ASR export:', asr_info)
PLAN = pipe.plan()
print('steps for', CONFIG['run_mode'], '->', PLAN + ['report'])

In [ ]:
# 6. PREPARE — audio manifests, external ASR import (schema inspection, canonical adapter), joins, integrity
if 'prepare' in PLAN:
    print(pipe._step('prepare', pipe.prepare))
st = pipe.store
schema = st.load_json('asr_import', 'asr_schema_report.json', {}) or {}
print('layout:', schema.get('layout'), '| roles:', schema.get('roles'), '| warnings:', schema.get('warnings'))
for sub, name in [('asr_import', 'asr_coverage.csv'), ('asr_import', 'import_gate.csv'), ('manifests', 'join_audit.csv')]:
    t = st.load_table(sub, name)
    if t is not None:
        print(f'--- {sub}/{name}')
        display(t.head(40))
integ = st.load_json('manifests', 'data_integrity.json', {}) or {}
print({k: integ.get(k) for k in ('n_train_clips', 'n_test_clips', 'n_train_students', 'n_test_students', 'n_fail', 'n_warn')})

In [ ]:
# 7. FEATURES — acoustic reuse / missing extraction, recognized text, disagreement, measured phonetics, providers
if 'features' in PLAN:
    print(pipe._step('features', pipe.features))
for sub, name in [('clip_features', 'acoustic_inventory.csv'), ('clip_features', 'acoustic_cache_reuse.csv'),
                  ('clip_features', 'phonetic_availability.csv')]:
    t = st.load_table(sub, name)
    if t is not None:
        print(f'--- {sub}/{name}')
        display(t)
cat = st.load_table('student_features', 'feature_catalog.parquet')
if cat is not None and len(cat):
    display(cat.groupby(['provider', 'family', 'modality', 'measurement']).size().rename('n_columns').reset_index())

In [ ]:
# 8. EXPERIMENTS — M0-M8, ablations, controls, robustness, fallback audit, H1-H10, importance, champion
if 'experiments' in PLAN:
    print(pipe._step('experiments', pipe.experiments))
cols = ['name', 'status', 'result_class', 'r2_mean', 'r2_sd', 'rmse', 'mae', 'delta_r2', 'delta_ci_low',
        'delta_ci_high', 'q', 'delta_r2_same_model', 'fallback_folds', 'n_added']
main = st.load_table('experiments', 'experiment_results.csv')
if main is not None:
    display(main[[c for c in cols if c in main.columns]])
fb = st.load_table('diagnostics', 'fallback_report.csv')
if fb is not None and 'is_fallback' in fb:
    print('--- rows that fell back to the baseline (NOT multimodal improvements)')
    display(fb[fb['is_fallback'] == True])
hyp = st.load_table('reports', 'hypotheses.csv')
if hyp is not None:
    display(hyp[[c for c in ('hypothesis', 'evidence', 'effect', 'ci_low', 'ci_high', 'experiment', 'result_class') if c in hyp]])
print('champion:', (st.load_json('model_registry', 'champion_spec.json', {}) or {}).get('name'))

In [ ]:
# 9. FIGURES — PNG + SVG + data tables under figures/
if 'figures' in PLAN:
    print(pipe._step('figures', pipe.figures))
idx = st.load_table('figures', 'figure_index.csv')
if idx is not None:
    display(idx[[c for c in ('figure', 'status', 'caption', 'n') if c in idx]])
    for p in idx.loc[idx['status'] == 'ok', 'png'].head(8):
        display(Image(filename=p))

In [ ]:
# 10. SUBMISSION — champion refit on all labelled students, re-load from disk, validated submission.csv
if 'submission' in PLAN:
    print(pipe._step('submission', pipe.submission))
val = st.load_table('submission', 'submission_validation.csv')
if val is not None:
    display(val)
sub = st.load_table('submission', 'submission.csv')
if sub is not None:
    display(sub.describe())

In [ ]:
# 11. REPORT + artefact inventory (everything is already on Drive under output_root)
report_path = pipe._step('report', pipe.report)
print('report:', report_path)
display(Markdown(open(report_path, encoding='utf-8').read()))
inv = st.load_table('reports', 'artifact_inventory.csv')
if inv is not None:
    display(inv)
display(pd.DataFrame(pipe.steps))